# CardScope: customer attrition investigation

This walkthrough uses the committed Kaggle snapshot and the same SQL as the dashboard. No machine learning is needed. Each row is one customer, not one transaction.

Source: [Credit Card Customers](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers). Provenance is not independently verified; read `data/README.md`.


In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pipeline import read_source, make_connection, query_outputs
rows, quality = read_source(ROOT / "data/raw/BankChurners.csv")
con = make_connection(rows)
print(f"Loaded {len(rows):,} customer records")
print(f"Retained columns: {quality['retained_columns']}")
print(f"Excluded model outputs: {len(quality['excluded_model_columns'])}")
print("Unknown values:", quality['unknown_categories'])


Loaded 10,127 customer records
Retained columns: 21
Excluded model outputs: 2
Unknown values: {'education_level': 1519, 'marital_status': 749, 'income_category': 1112}


## 1. Establish the baseline

The numerator is customers labelled attrited. The denominator is all customers in the snapshot. There is no date field, so this is not a monthly churn calculation.


In [2]:
sql = (ROOT / "sql/02_overview.sql").read_text()
print(sql)
print(json.dumps(dict(con.execute(sql).fetchone()), indent=2))


-- Attrition is an observed snapshot label, not a monthly churn event rate.
SELECT COUNT(*) AS customers,
       SUM(churned) AS churned,
       COUNT(*) - SUM(churned) AS retained,
       1.0 * SUM(churned) / NULLIF(COUNT(*), 0) AS churn_rate,
       AVG(total_trans_ct) AS avg_transactions,
       AVG(total_trans_amt) AS avg_transaction_amount,
       AVG(avg_utilization_ratio) AS avg_utilization
FROM customer_features;

{
  "customers": 10127,
  "churned": 1627,
  "retained": 8500,
  "churn_rate": 0.1606596227905599,
  "avg_transactions": 64.85869457884863,
  "avg_transaction_amount": 4404.086303939963,
  "avg_utilization": 0.2748935518909845
}


## 2. Separate segment rate from contribution to total attrition

A small segment can have a high rate while contributing few customers. Always show both counts and denominators. Below, all transaction bands sum to the full population.


In [3]:
result = query_outputs(con)
for group in result['segments']:
    if group['dimension'] == 'transaction_band':
        print(f"{group['segment']:>5}: n={group['customers']:4d}, attrited={group['churned']:3d}, "
              f"rate={group['churn_rate']:.2%}, share of attrited={group['share_of_churn']:.2%}")
print("Population reconciliation:", sum(g['customers'] for g in result['segments']
                                         if g['dimension'] == 'transaction_band'))


40-59: n=2134, attrited=835, rate=39.13%, share of attrited=51.32%
  <40: n=1802, attrited=542, rate=30.08%, share of attrited=33.31%
60-79: n=3468, attrited=214, rate=6.17%, share of attrited=13.15%
  80+: n=2723, attrited= 36, rate=1.32%, share of attrited=2.21%
Population reconciliation: 10127


## 3. Compare behavior, without asserting causes

The contact field does not distinguish complaints from ordinary contacts. Transaction activity could decline before or after attrition; this snapshot cannot establish the sequence.


In [4]:
for group in result['behavior']:
    print(group['attrition_flag'])
    print(f"  Avg transactions: {group['avg_transactions']:.2f}")
    print(f"  Avg inactive months: {group['avg_inactive_months']:.2f}")
    print(f"  Avg recorded contacts: {group['avg_contacts']:.2f}")


Attrited Customer
  Avg transactions: 44.93
  Avg inactive months: 2.69
  Avg recorded contacts: 2.97
Existing Customer
  Avg transactions: 68.67
  Avg inactive months: 2.27
  Avg recorded contacts: 2.36


## 4. Explore a transparent investigation cohort

The rule `<60 transactions AND >=3 inactive months` is a hand-chosen retrospective segmentation example. It is not a validated prediction rule. Do not export it as a ready-to-contact customer list.


In [5]:
for group in result['cohorts']:
    print(f"{group['cohort']}: n={group['customers']:,}, attrited={group['churned']:,}, "
          f"existing={group['existing_customers']:,}, rate={group['churn_rate']:.2%}")


Investigate: low activity + inactivity: n=1,903, attrited=848, existing=1,055, rate=44.56%
Other customers: n=8,224, attrited=779, existing=7,445, rate=9.47%


## 5. Check a filtered result independently

Change `Blue` and `Unknown` and compare the result with the dashboard. Parameters avoid unsafe string interpolation in SQL.


In [6]:
filtered = con.execute("""
SELECT COUNT(*) AS customers, SUM(churned) AS attrited,
       AVG(churned) AS attrition_proportion
FROM customer_features
WHERE card_category = ? AND income_category = ?
""", ("Blue", "Unknown")).fetchone()
print(dict(filtered))
con.close()


{'customers': 1045, 'attrited': 176, 'attrition_proportion': 0.16842105263157894}


## Recommendation

Request dated activity and closure histories, clarify contact definitions, and validate that features precede outcomes. Only then consider a randomized pilot with pre-defined eligibility, 90-day retention, costs and opt-out guardrails.

No fee effects, causal drivers, campaign uplift or ROI have been demonstrated. See `reports/analysis.md` for the evidence and `docs/business-analysis.md` for the proposed requirements.
